# ครั้งที่ 5 — Floating-point (IEEE 754)
**51423364 Embedded System Programming | ศุกร์ 7 ส.ค. 2569**

ครั้งที่แล้วเราเก็บทศนิยมด้วย **fixed-point** (จุดทศนิยมอยู่กับที่)
วันนี้มาดูอีกวิธี: **floating-point** — จุดทศนิยม "เลื่อนได้" ตามขนาดของค่า

> กด `Shift + Enter` รันทีละเซลล์


## 1. แนวคิด: เลขวิทยาศาสตร์ในฐานสอง
เลขฐานสิบเขียนเป็น 3.75 = 3.75 × 10⁰ ได้ฉันใด
เลขฐานสองก็เขียนเป็น 1.xxx × 2ᵉ ได้ฉันนั้น (เรียกว่า normalized form)

IEEE 754 แบบ 32 บิต (float) แบ่งเป็น 3 ส่วน:

| ส่วน | บิต | ความหมาย |
|---|---|---|
| sign | 1 | 0 = บวก, 1 = ลบ |
| exponent | 8 | เลขชี้กำลัง เก็บแบบ bias 127 |
| mantissa | 23 | ส่วนเศษ (บิตนำหน้า 1. ถูกซ่อนไว้) |


In [1]:
import struct

def show_float(x):
    bits = format(int.from_bytes(struct.pack('>f', x), 'big'), '032b')
    s, e, m = bits[0], bits[1:9], bits[9:]
    print(f"{x}")
    print(f"  sign     = {s}")
    print(f"  exponent = {e}  -> {int(e,2)} - 127 = {int(e,2)-127}")
    print(f"  mantissa = {m}")
    print()

show_float(1.0)
show_float(3.75)
show_float(-2.5)

1.0
  sign     = 0
  exponent = 01111111  -> 127 - 127 = 0
  mantissa = 00000000000000000000000

3.75
  sign     = 0
  exponent = 10000000  -> 128 - 127 = 1
  mantissa = 11100000000000000000000

-2.5
  sign     = 1
  exponent = 10000000  -> 128 - 127 = 1
  mantissa = 01000000000000000000000



**ตรวจสอบด้วยมือ: 3.75**
- 3.75 ในฐานสอง = `11.11`
- normalize → `1.111 × 2¹` → exponent = 1 → เก็บเป็น 1 + 127 = 128 = `10000000` ✓
- mantissa เก็บเฉพาะส่วนหลังจุด = `111` แล้วเติม 0 ให้ครบ 23 บิต ✓

### ลองเอง 1
ใช้ `show_float()` ดูค่า `0.15625` แล้วตรวจว่า exponent ควรเป็น −3 หรือไม่

In [2]:
show_float(0.15625)   # ?

0.15625
  sign     = 0
  exponent = 01111100  -> 124 - 127 = -3
  mantissa = 01000000000000000000000



## 2. ช่วงค่าและความละเอียด — จุดต่างจาก fixed-point

In [3]:
import sys
print("float32 (C: float)")
print("  ค่าบวกน้อยสุด (normal):", struct.unpack('>f', bytes.fromhex('00800000'))[0])
print("  ค่ามากสุด             :", struct.unpack('>f', bytes.fromhex('7f7fffff'))[0])
print("  ความแม่นยำ            : ~7 หลักฐานสิบ (mantissa 24 บิตรวมบิตซ่อน)")
print()
print("เทียบกับ Q8.8 (16 บิต): ช่วง -128 ถึง 127.996 ละเอียดคงที่ 0.0039 เสมอ")

float32 (C: float)
  ค่าบวกน้อยสุด (normal): 1.1754943508222875e-38
  ค่ามากสุด             : 3.4028234663852886e+38
  ความแม่นยำ            : ~7 หลักฐานสิบ (mantissa 24 บิตรวมบิตซ่อน)

เทียบกับ Q8.8 (16 บิต): ช่วง -128 ถึง 127.996 ละเอียดคงที่ 0.0039 เสมอ


**ใจความสำคัญ:** fixed-point ความละเอียด **คงที่** ทุกช่วงค่า
ส่วน float ความละเอียด **เปลี่ยนตามขนาดค่า** — ค่าน้อยละเอียดมาก ค่าใหญ่หยาบลง

In [4]:
for v in [1.0, 1000.0, 1000000.0]:
    nxt = struct.unpack('>f', struct.pack('>I', int.from_bytes(struct.pack('>f', v),'big') + 1))[0]
    print(f"ที่ค่า {v:>12,.0f}  ค่าถัดไปที่แทนได้ห่างกัน {nxt - v}")

ที่ค่า            1  ค่าถัดไปที่แทนได้ห่างกัน 1.1920928955078125e-07
ที่ค่า        1,000  ค่าถัดไปที่แทนได้ห่างกัน 6.103515625e-05
ที่ค่า    1,000,000  ค่าถัดไปที่แทนได้ห่างกัน 0.0625


### ลองเอง 2
จากผลด้านบน — ถ้าเอา float มานับเวลาเป็นวินาทีสะสมนาน ๆ จะเกิดปัญหาอะไร?

In [ ]:
for v in range(1, 60):
    nxt = struct.unpack('>f', struct.pack('>I', int.from_bytes(struct.pack('>f', v),'big') + 1))[0]
    print(f"ที่ค่า {v:>12,.0f}  ค่าถัดไปที่แทนได้ห่างกัน {nxt - v}")

#ค่าที่โตขึ้นเรื่อยๆ อย่างไม่มีขอบเขต เพราะความละเอียดจะลดลงตามขนาดของค่า

ที่ค่า            1  ค่าถัดไปที่แทนได้ห่างกัน 1.1920928955078125e-07
ที่ค่า            2  ค่าถัดไปที่แทนได้ห่างกัน 2.384185791015625e-07
ที่ค่า            3  ค่าถัดไปที่แทนได้ห่างกัน 2.384185791015625e-07
ที่ค่า            4  ค่าถัดไปที่แทนได้ห่างกัน 4.76837158203125e-07
ที่ค่า            5  ค่าถัดไปที่แทนได้ห่างกัน 4.76837158203125e-07
ที่ค่า            6  ค่าถัดไปที่แทนได้ห่างกัน 4.76837158203125e-07
ที่ค่า            7  ค่าถัดไปที่แทนได้ห่างกัน 4.76837158203125e-07
ที่ค่า            8  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า            9  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           10  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           11  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           12  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           13  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           14  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07
ที่ค่า           15  ค่าถัดไปที่แทนได้ห่างกัน 9.5367431640625e-07

## 3. ปัญหาคลาสสิก: 0.1 + 0.2 != 0.3

In [5]:
a = 0.1 + 0.2
print("0.1 + 0.2 =", repr(a))
print("เท่ากับ 0.3 ไหม?", a == 0.3)
print()
# ทำไม? เพราะ 0.1 ในฐานสองเป็นทศนิยมไม่รู้จบ
print("0.1 เก็บใน float32 ได้จริง ๆ =", struct.unpack('>f', struct.pack('>f', 0.1))[0])
print("hex ของ 0.1 (float32)       =", struct.pack('>f', 0.1).hex())

0.1 + 0.2 = 0.30000000000000004
เท่ากับ 0.3 ไหม? False

0.1 เก็บใน float32 ได้จริง ๆ = 0.10000000149011612
hex ของ 0.1 (float32)       = 3dcccccd


**เหมือนกับ 1/3 ในฐานสิบ = 0.3333... ไม่มีวันจบ**
0.1 ในฐานสองก็เป็น 0.0001100110011... ไม่มีวันจบเช่นกัน

**วิธีที่ถูกต้องในการเทียบ float:**

In [6]:
EPS = 1e-9
print("เทียบตรง ๆ :", 0.1 + 0.2 == 0.3)
print("เทียบด้วย EPS:", abs((0.1 + 0.2) - 0.3) < EPS)   # แบบนี้ถูกต้อง

เทียบตรง ๆ : False
เทียบด้วย EPS: True


### ลองเอง 3
ลองบวก 0.1 สะสม 1000 ครั้งด้วย float แล้วเทียบกับ fixed-point Q8.8 จากครั้งที่แล้ว

In [7]:
acc = 0.0
for _ in range(1000):
    acc += 0.1
print("float  :", acc)

# fixed-point Q8.8 จากครั้งที่แล้ว
q = round(0.1 * 256)
print("Q8.8   :", (q * 1000) / 256)
print("ค่าจริง :", 100)

float  : 99.9999999999986
Q8.8   : 101.5625
ค่าจริง : 100


## 4. ค่าพิเศษของ IEEE 754

In [8]:
inf = struct.unpack('>f', bytes.fromhex('7f800000'))[0]
nan = struct.unpack('>f', bytes.fromhex('7fc00000'))[0]
print("infinity :", inf, "| exponent เป็น 1 ทั้งหมด, mantissa = 0")
print("NaN      :", nan, "| exponent เป็น 1 ทั้งหมด, mantissa != 0")
print()
print("1.0 / 0.0 ใน C ได้ inf (ไม่ crash) แต่ Python จะ raise error:")
try:
    print(1.0 / 0.0)
except ZeroDivisionError as e:
    print("  ZeroDivisionError:", e)
print()
print("NaN แปลก ๆ ตรงที่ไม่เท่ากับตัวเอง:", nan == nan)

infinity : inf | exponent เป็น 1 ทั้งหมด, mantissa = 0
NaN      : nan | exponent เป็น 1 ทั้งหมด, mantissa != 0

1.0 / 0.0 ใน C ได้ inf (ไม่ crash) แต่ Python จะ raise error:
  ZeroDivisionError: division by zero

NaN แปลก ๆ ตรงที่ไม่เท่ากับตัวเอง: False


## 5. เลือก fixed หรือ float ดี?

| ประเด็น | Fixed-point | Floating-point |
|---|---|---|
| ความเร็วบน MCU ไม่มี FPU | เร็ว (คำนวณจำนวนเต็ม) | ช้ามาก (จำลองด้วยซอฟต์แวร์) |
| ขนาดโค้ด | เล็ก | ใหญ่ (ต้องลิงก์ไลบรารี) |
| ช่วงค่า | แคบ กำหนดเอง | กว้างมาก (10³⁸) |
| ความละเอียด | คงที่ทุกช่วง | เปลี่ยนตามขนาดค่า |
| เหมาะกับ | เซนเซอร์, ควบคุม, DSP บน MCU เล็ก | คำนวณวิทยาศาสตร์, MCU ที่มี FPU |

**ESP32 มี FPU (single precision)** — ใช้ `float` ได้สบาย แต่ `double` ยังช้าอยู่
ส่วน Arduino UNO (AVR) ไม่มี FPU — ควรเลี่ยง float ถ้าเป็นไปได้

### ลองเอง 4
เขียนโค้ดเทียบเวลาบวกเลข 1 ล้านครั้งระหว่าง int กับ float

In [9]:
import time

t0 = time.perf_counter()
s = 0
for i in range(1_000_000):
    s += i
t_int = time.perf_counter() - t0

t0 = time.perf_counter()
f = 0.0
for i in range(1_000_000):
    f += i * 0.5
t_flt = time.perf_counter() - t0

print(f"int   : {t_int:.4f} s")
print(f"float : {t_flt:.4f} s")
print("หมายเหตุ: บน PC ต่างกันไม่มาก แต่บน MCU ที่ไม่มี FPU ต่างกันหลายสิบเท่า")

int   : 0.0998 s
float : 0.0827 s
หมายเหตุ: บน PC ต่างกันไม่มาก แต่บน MCU ที่ไม่มี FPU ต่างกันหลายสิบเท่า


---
## เตรียมตัวครั้งที่ 6 (14 ส.ค.) — เริ่มเขียนภาษา C จริง!

สิ่งที่เรียนมา 3 ครั้ง (เลขฐาน, fixed-point, floating-point) จะได้ใช้จริงในภาษา C
- ทบทวน `uint8_t`, `int8_t`, `float` ว่าต่างกันอย่างไร
- เตรียม VS Code + Notebook ให้พร้อม (จะเริ่มเขียน C ใน Arduino IDE)
